<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-engineer-june20/blob/main/Day_12_PDF_QA_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 12 — Lab: Build a PDF Q&A App With Claude
### AI Architect Mastery Program · Lab Session · Level: Beginner-friendly

**Yesterday in 4 lines:** Meera at NammaPay needed Claude to answer from a 48-page
handbook it has never read. We learned the fix — RAG: `chunk → embed → store →
retrieve → generate`, with a grounding prompt so Claude answers *only* from the
document. Today we stop talking and build it — on a **real PDF you choose**.

---

## What you'll build in the next ~90 minutes

1. 📄 **Upload any PDF** → extract and clean its text
2. ✂️ **Chunk** it with overlap → 🗺️ **embed** with a real model → 🗄️ **store** in ChromaDB
3. 🤖 **Ask Claude Haiku questions** → grounded, cited answers
4. 📊 **The experiment:** index the same PDF at **3 chunk sizes** and print a scorecard —
   yesterday's claim ("chunk size is the sharpest knob"), measured
5. 🛡️ **Grounding test:** prove the app says "I don't know" instead of making things up

No mock code. Every Claude call is the real API. By the end you'll have a working
app *and* numbers you can defend in a design review.

| Flag | Meaning |
|---|---|
| 💻 run it | 📊 how to read the result | 🐛 gotcha | 🧪 try this | ⚠️ accuracy note | 🎤 interview angle |

## Step 0 — Install (one cell, ~1 minute)

Five libraries, each with one job:

| Library | Job in our pipeline |
|---|---|
| `pdfplumber` | pull text out of the PDF |
| `sentence-transformers` | the embedding model (meaning map) |
| `chromadb` | the vector database (smart filing cabinet) |
| `anthropic` | talk to Claude Haiku |
| `fpdf2` | (optional) create a sample PDF if you don't have one handy |

In [2]:
!pip install -q pdfplumber sentence-transformers chromadb anthropic fpdf2
print("All libraries installed ✅")

All libraries installed ✅


## Step 1 — API key + Claude client

> In Colab: 🔑 **Secrets** panel (left sidebar) → add `MY_API_KEY` = your Anthropic key
> → toggle "Notebook access" ON.

In [3]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"   # fast + cheap — ideal for a Q&A app

print("Claude client ready ✅")

Claude client ready ✅


## Step 2 — Get a PDF

**Option A (recommended):** upload your own — a manual, a syllabus, a policy, a report.
Best results with a **text PDF** of 5+ pages (if you can select text in a PDF viewer,
it's a text PDF; scanned photocopies won't work — see the gotcha below).

**Option B:** no PDF handy? Skip the upload cell and run the *sample generator* cell —
it creates Meera's NammaPay handbook as a real multi-page PDF, so the whole lab works
out of the box.

In [4]:
# Option A — upload your own PDF
from google.colab import files

print("A file picker will appear — choose a PDF.")
uploaded = files.upload()

PDF_PATH = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {PDF_PATH} ({len(uploaded[PDF_PATH])/1024:.0f} KB)")

A file picker will appear — choose a PDF.


Saving Arul_Benjamin_Chandru_Mainframe_Resume.pdf to Arul_Benjamin_Chandru_Mainframe_Resume (1).pdf

✅ Uploaded: Arul_Benjamin_Chandru_Mainframe_Resume (1).pdf (8 KB)


In [18]:
# Option B — OR generate the sample NammaPay handbook (run only if you skipped Option A)
from fpdf import FPDF

SECTIONS = [
    ("1. Settlements", "NammaPay settles merchant payments on a T+1 basis: money collected on "
     "Monday reaches your bank account on Tuesday. UPI settlements are free of charge. Card "
     "settlements carry a fee of 1.9 percent. Settlements are paused if KYC is incomplete. "
     "Merchants can track every settlement in the dashboard under Payments, then Settlements."),
    ("2. Refunds", "If a customer's payment fails but money is deducted, the amount is refunded "
     "automatically within 7 days by the bank. Merchants can issue manual refunds from the "
     "dashboard within 90 days of the original transaction. Refunds are always free: NammaPay "
     "charges no fee, and the original transaction fee is returned to the merchant as well."),
    ("3. Disputes and chargebacks", "When a customer disputes a UPI transaction, the merchant "
     "must respond with evidence within 5 working days. Evidence can include delivery proof, "
     "invoices, or customer communication. If no response is received in time, the dispute is "
     "decided in the customer's favour automatically. Each lost chargeback carries a fee of "
     "Rs 250. Merchants with more than 20 lost disputes per month may be suspended."),
    ("4. KYC requirements", "Every merchant must complete KYC verification with a PAN card and "
     "one address proof such as an Aadhaar card, passport, or utility bill. Companies must also "
     "provide a certificate of incorporation. Accounts with incomplete KYC cannot receive "
     "settlements above Rs 50,000 per month and cannot use international payments."),
    ("5. Merchant support", "Merchant support is available from 9am to 9pm IST, seven days a "
     "week, through the chat option in the NammaPay dashboard. Critical payment outages are "
     "handled 24x7 through the emergency hotline listed in the dashboard. The average first "
     "response time for chat is under 4 minutes."),
    ("6. Fees summary", "UPI payments: zero fees. Debit and credit cards: 1.9 percent per "
     "transaction. International cards: 3.5 percent per transaction. Chargeback fee: Rs 250 "
     "per lost dispute. Instant settlement (optional add-on): 0.2 percent extra. There are no "
     "setup charges, no annual charges, and no hidden fees of any kind."),
]

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=18)
for title, body in SECTIONS:
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14); pdf.multi_cell(0, 8, f"NammaPay Merchant Policy Handbook - {title}")
    pdf.ln(3)
    pdf.set_font("Helvetica", size=11); pdf.multi_cell(0, 7, body * 3)   # repeat so pages have substance

pdf.output("nammapay_handbook.pdf")
PDF_PATH = "nammapay_handbook.pdf"
print(f"✅ Sample handbook created: {PDF_PATH} (6 pages)")

✅ Sample handbook created: nammapay_handbook.pdf (6 pages)


## Step 3 — Extract and clean the text

💻 `pdfplumber` reads each page; then three small cleanups: collapse repeated spaces,
repair words broken by end-of-line hyphens, and normalise blank lines.

In [19]:
import pdfplumber
import re

def extract_and_clean(pdf_path):
    """Extract text from every page of a PDF and lightly clean it."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        print(f"PDF has {len(pdf.pages)} pages")
        for i, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()
            if text:
                pages.append(text)
                print(f"  page {i}: {len(text)} chars")
            else:
                print(f"  page {i}: NO TEXT (scanned image? see gotcha below)")

    full = "\n\n".join(pages)
    full = re.sub(r"(\w)-\n(\w)", r"\1\2", full)   # re-join words hyphen-\n-broken across lines
    full = re.sub(r"[ \t]+", " ", full)              # collapse runs of spaces
    full = re.sub(r"\n{3,}", "\n\n", full)          # collapse big blank gaps
    return full.strip()

DOCUMENT = extract_and_clean(PDF_PATH)

print(f"\n✅ Extracted {len(DOCUMENT):,} characters ≈ {len(DOCUMENT)//4:,} tokens")
print("\nFirst 300 characters:\n" + "-"*50)
print(DOCUMENT[:300])

PDF has 6 pages
  page 1: 1035 chars
  page 2: 1049 chars
  page 3: 1249 chars
  page 4: 1004 chars
  page 5: 890 chars
  page 6: 964 chars

✅ Extracted 6,201 characters ≈ 1,550 tokens

First 300 characters:
--------------------------------------------------
NammaPay Merchant Policy Handbook - 1. Settlements
NammaPay settles merchant payments on a T+1 basis: money collected on Monday reaches your bank
account on Tuesday. UPI settlements are free of charge. Card settlements carry a fee of 1.9 percent.
Settlements are paused if KYC is incomplete. Merchant


🐛 **Gotcha — the silent killer of PDF pipelines:** a *scanned* PDF is photographs of
pages; `extract_text()` returns `None` and your pipeline continues happily **with
nothing in it** — no error anywhere. That's why we print per-page character counts:
a run of "NO TEXT" pages means you need OCR (e.g. Tesseract) before RAG.
In production, alert whenever a document extracts to near-zero characters.

📊 Also note the **≈ token estimate** — this decides everything downstream:
a 20,000-token document at chunk size 400 → about 50–60 chunks (overlap adds a few).

## Step 4 — Chunk the document

✂️ Sentence-aware chunking with overlap (Day 11, Strategy B + the overlap trick):
pack whole sentences until the chunk is full; start the next chunk by *re-using the
last sentence* of the previous one, so nothing is orphaned at a boundary.

The function takes `chunk_tokens` as a parameter — **that's deliberate**: in Step 8
we'll call it with three different sizes and measure what changes.

In [20]:
import re

def chunk_document(text, chunk_tokens=400, overlap_sentences=1):
    """Split text into chunks of whole sentences, ~chunk_tokens each.
    Consecutive chunks share the last `overlap_sentences` sentences (the overlap)."""
    max_chars = chunk_tokens * 4                       # rule of thumb: 1 token ≈ 4 chars
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s for s in sentences if s.strip()]

    chunks, current = [], []
    size = 0
    for s in sentences:
        if size + len(s) > max_chars and current:
            chunks.append(" ".join(current))
            current = current[-overlap_sentences:]      # carry the tail forward = overlap
            size = sum(len(x) for x in current)
        current.append(s)
        size += len(s)
    if current:
        chunks.append(" ".join(current))
    return chunks

CHUNKS = chunk_document(DOCUMENT, chunk_tokens=400)

print(f"✅ {len(CHUNKS)} chunks (target ≈400 tokens each)")
avg = sum(len(c) for c in CHUNKS) // max(len(CHUNKS), 1)
print(f"   average size: {avg} chars ≈ {avg//4} tokens")
print(f"\nChunk 0 preview:\n" + "-"*50 + f"\n{CHUNKS[0][:350]}...")

✅ 5 chunks (target ≈400 tokens each)
   average size: 1323 chars ≈ 330 tokens

Chunk 0 preview:
--------------------------------------------------
NammaPay Merchant Policy Handbook - 1. Settlements
NammaPay settles merchant payments on a T+1 basis: money collected on Monday reaches your bank
account on Tuesday. UPI settlements are free of charge. Card settlements carry a fee of 1.9 percent. Settlements are paused if KYC is incomplete. Merchants can track every settlement in the dashboard unde...


## Step 5 — Embed every chunk

🗺️ `all-MiniLM-L6-v2`: free, runs locally, 384 dimensions, ~90 MB download on first
run. Small enough to be fast in Colab, good enough to prove every concept.
(Production upgrade path: `all-mpnet-base-v2` or Voyage AI — same code shape.)

In [7]:
!pip install --upgrade Pillow torchvision
print("Upgraded Pillow and torchvision ✅")

Upgraded Pillow and torchvision ✅


In [21]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model (downloads ~90 MB on first run)...")
EMBEDDER = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✅ Model ready — {EMBEDDER.get_sentence_embedding_dimension()} dimensions per text")

CHUNK_VECTORS = EMBEDDER.encode(CHUNKS, show_progress_bar=True)
print(f"\n✅ Embedded {CHUNK_VECTORS.shape[0]} chunks → matrix shape {CHUNK_VECTORS.shape}")

Loading embedding model (downloads ~90 MB on first run)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model ready — 384 dimensions per text


/tmp/ipykernel_5057/256350619.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Model ready — {EMBEDDER.get_sentence_embedding_dimension()} dimensions per text")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Embedded 5 chunks → matrix shape (5, 384)


📊 **How to read this:** shape `(N, 384)` = N chunks, each now a pin with 384
coordinates on the meaning map. This is the *offline* half of the pipeline doing
its one-time work.

## Step 6 — Store in ChromaDB

🗄️ Each record = text + vector + metadata. Cosine space set **explicitly**
(remember Day 11's accuracy note: Chroma defaults to L2, silently).

In [22]:
import chromadb

CHROMA = chromadb.Client()      # in-memory: perfect for a lab (disappears on restart)

def build_collection(name, chunks, vectors):
    """Create a cosine-space collection and load chunks + vectors + metadata into it."""
    try:
        CHROMA.delete_collection(name)          # start fresh if the cell is re-run
    except Exception:
        pass
    col = CHROMA.create_collection(name=name, metadata={"hnsw:space": "cosine"})
    col.add(
        ids=[f"{name}_{i}" for i in range(len(chunks))],
        documents=list(chunks),
        embeddings=vectors.tolist(),
        metadatas=[{"chunk_index": i, "source": PDF_PATH} for i in range(len(chunks))],
    )
    return col

COLLECTION = build_collection("pdf_chunks_400", CHUNKS, CHUNK_VECTORS)
print(f"✅ Stored {COLLECTION.count()} chunks in ChromaDB (cosine space)")

✅ Stored 5 chunks in ChromaDB (cosine space)


## Step 7 — Retrieve, then ask Claude (the app comes alive)

Two functions, then you have a working PDF Q&A app:

- `retrieve(question, collection, k)` — embed the question (same model!), return the
  top-k chunks with similarity scores
- `ask_pdf(question, ...)` — retrieve → build a numbered context block → grounded
  Claude Haiku call → answer with citations

In [23]:
def retrieve(question, collection=None, k=3):
    """Return the k chunks most similar in meaning to the question."""
    collection = collection or COLLECTION
    qvec = EMBEDDER.encode([question])
    res = collection.query(query_embeddings=qvec.tolist(), n_results=k)
    return [
        {"text": doc, "similarity": round(1 - dist, 3), "chunk_index": meta["chunk_index"]}
        for doc, dist, meta in zip(res["documents"][0], res["distances"][0], res["metadatas"][0])
    ]

# quick look at retrieval on its own
for r in retrieve("What is this document about?", k=3):
    print(f"chunk #{r['chunk_index']}  similarity {r['similarity']}  | {r['text'][:90]}...")

chunk #0  similarity 0.06  | NammaPay Merchant Policy Handbook - 1. Settlements
NammaPay settles merchant payments on a...
chunk #3  similarity 0.04  | NammaPay Merchant Policy Handbook - 5. Merchant support
Merchant support is available from...
chunk #2  similarity 0.038  | Each lost chargeback carries a fee of Rs 250. Merchants with more
than 20 lost disputes pe...


In [24]:
GROUNDING_PROMPT = """You are a document Q&A assistant.

Rules:
1. Answer ONLY from the context below. Ignore your general knowledge.
2. If the context does not contain the answer, reply exactly:
   "I don't have this information in the document."
3. Cite the chunk number, like [Chunk 4], for each fact you use.
4. Be concise.

Context:
{context}"""

def ask_pdf(question, collection=None, k=3, show_chunks=True):
    """Full RAG query: retrieve -> grounded Claude Haiku call -> cited answer."""
    hits = retrieve(question, collection, k)
    context = "\n\n".join(f"[Chunk {h['chunk_index']}]\n{h['text']}" for h in hits)

    reply = client.messages.create(
        model=MODEL,
        max_tokens=400,
        system=GROUNDING_PROMPT.format(context=context),
        messages=[{"role": "user", "content": question}],
    )

    print(f"Q: {question}")
    print(f"A: {reply.content[0].text}")
    print(f"   [tokens in: {reply.usage.input_tokens}, out: {reply.usage.output_tokens}]")
    if show_chunks:
        used = ", ".join(f"#{h['chunk_index']}({h['similarity']})" for h in hits)
        print(f"   [retrieved: {used}]")
    print()
    return reply.content[0].text

print("✅ PDF Q&A app ready — ask_pdf('your question')")

✅ PDF Q&A app ready — ask_pdf('your question')


## Your first questions

💻 Edit the questions to match *your* PDF and run:

In [25]:
ask_pdf("What is this document about?")
ask_pdf("Summarise the most important rule or finding in this document.")

Q: What is this document about?
A: This document is the **NammaPay Merchant Policy Handbook** [Chunk 0]. It covers NammaPay's policies and procedures for merchants, including:

1. **Settlements** - Payment settlement timelines and fees [Chunk 0]
2. **Refunds** - Automatic and manual refund processes [Chunk 0]
3. **Disputes/Chargebacks** - Chargeback procedures and fees [Chunk 2]
4. **KYC Requirements** - Verification documentation needed [Chunk 2]
5. **Merchant Support** - Available support channels and hours [Chunk 3]
6. **Fees Summary** - Complete fee structure for different payment types [Chunk 3]

The handbook outlines the rules, fees, and procedures merchants need to know to use NammaPay's payment processing services.
   [tokens in: 1285, out: 196]
   [retrieved: #0(0.06), #3(0.04), #2(0.038)]

Q: Summarise the most important rule or finding in this document.
A: Based on the document, the most important rule is:

**NammaPay settles merchant payments on a T+1 basis (next business d

'Based on the document, the most important rule is:\n\n**NammaPay settles merchant payments on a T+1 basis (next business day), with different fees depending on payment method** [Chunk 0]. Specifically:\n- UPI payments are free\n- Card payments cost 1.9% [Chunk 0]\n- International cards cost 3.5% [Chunk 3]\n\nHowever, **settlements are paused if KYC verification is incomplete** [Chunk 0], and incomplete KYC accounts cannot receive settlements above Rs 50,000 per month [Chunk 2]. This makes KYC completion a critical prerequisite for any merchant to receive payments.'

📊 **How to read this:** check three things — the answer is *plausible for your PDF*;
it carries **[Chunk N] citations**; and the token counts are small (that's RAG's cost
advantage: a few hundred input tokens instead of the whole document every time).

🧪 **Try this:** open your PDF, find the cited chunk's text, and verify the claim
yourself. That checkability *is* the product. Then ask something tricky: a question
whose answer spans two sections, or uses different words than the document.

🐛 **Gotcha:** re-running the ChromaDB cell without deleting the collection would
error ("collection already exists") or double-insert — that's why `build_collection`
deletes first. In real apps you'd *upsert* by stable IDs instead.

## Optional — free-form Q&A loop

An interactive session: type questions, `quit` to stop.

In [26]:
while True:
    q = input("Ask your PDF (or 'quit'): ").strip()
    if q.lower() in ("quit", "exit", ""):
        print("Session ended.")
        break
    ask_pdf(q)

Ask your PDF (or 'quit'): what is the waiting time to raise a dispute on upi fraud?
Q: what is the waiting time to raise a dispute on upi fraud?
A: I don't have this information in the document.

The document [Chunk 1] mentions that when a customer disputes a UPI transaction, the merchant must respond with evidence within 5 working days, but it does not specify a waiting time or deadline for raising a dispute in the first place.
   [tokens in: 1231, out: 65]
   [retrieved: #1(0.423), #2(0.406), #0(0.393)]

Ask your PDF (or 'quit'): what happens when a customer disputes UPI?
Q: what happens when a customer disputes UPI?
A: When a customer disputes a UPI transaction, the following happens [Chunk 1]:

1. **Merchant must respond with evidence within 5 working days** - Evidence can include delivery proof, invoices, or customer communication.

2. **Automatic decision if no response** - If no response is received in time, the dispute is decided in the customer's favor automatically.

3. **Fee

---
# Step 8 — The Experiment: 3 Chunk Sizes, 1 Scorecard 🥊

Yesterday's claim: *"chunk size is the sharpest knob in RAG."* Today we don't repeat
the claim — we test it. Same PDF, same questions, same embedding model, same scoring —
only the chunk size changes:

| Contender | Chunk size | Personality |
|---|---|---|
| **Small** | 150 tokens | laser-focused index cards |
| **Medium** | 400 tokens | the textbook default |
| **Large** | 800 tokens | fat paragraphs with lots of context |

**Fair-fight rules** (this is what makes it an experiment, not a demo):
one evaluation set used for all three; one scoring function used for all three.
If we scored each size differently, the comparison would mean nothing.

### First: build all three indexes

In [27]:
CHUNK_SIZES = [150, 400, 800]
INDEXES = {}

for size in CHUNK_SIZES:
    chunks  = chunk_document(DOCUMENT, chunk_tokens=size)
    vectors = EMBEDDER.encode(chunks, show_progress_bar=False)
    # NOTE: fresh names ("exp_..."), so the main COLLECTION from Step 6 stays valid —
    # build_collection deletes-and-recreates, which would orphan the old object.
    INDEXES[size] = build_collection(f"exp_{size}", chunks, vectors)
    avg = sum(len(c) for c in chunks) // max(len(chunks), 1)
    print(f"chunk_tokens={size:>4}: {len(chunks):>4} chunks, avg {avg//4} tokens each")

chunk_tokens= 150:   13 chunks, avg 137 tokens each
chunk_tokens= 400:    5 chunks, avg 330 tokens each
chunk_tokens= 800:    2 chunks, avg 786 tokens each


📊 **How to read this:** smaller chunk size → *more* chunks (same text, thinner
slices). Already visible: at 150 tokens your PDF becomes several times more pieces
than at 800.

### Second: your evaluation set

✍️ **Edit this cell** — write 5 questions about *your* PDF, and for each one a short
`expect` phrase that must appear in the chunk containing the answer (copy the exact
wording from the PDF; matching is case-insensitive). The defaults work for the sample
NammaPay handbook.

In [28]:
# question  = what a user would ask (user language)
# expect    = a phrase from the PDF that the correct chunk must contain (document language)
EVAL_SET = [
    {"question": "How fast do merchants receive their settlement money?",
     "expect": "T+1"},
    {"question": "What happens if I ignore a customer dispute?",
     "expect": "customer's favour"},
    {"question": "How much does a lost chargeback cost?",
     "expect": "250"},
    {"question": "Which ID documents are needed to verify my account?",
     "expect": "PAN card"},
    {"question": "What are the charges for international card payments?",
     "expect": "3.5"},
]
print(f"✅ {len(EVAL_SET)} evaluation questions defined")

✅ 5 evaluation questions defined


Notice the questions deliberately use *different words* than the document
("settlement money" vs "T+1 basis", "ignore" vs "no response received") — that's
realistic: users never phrase questions the way documents phrase answers. Embeddings
are supposed to bridge that gap; now we check whether they do at each chunk size.

### Third: one scoring function, one scorecard

In [29]:
def score_index(collection, eval_set, k=3):
    """For each question: did any of the top-k retrieved chunks contain the expected
    phrase? Returns hit-rate, average top-1 similarity, and avg context size sent."""
    hits, top1_sims, context_chars = 0, [], []
    misses = []
    for item in eval_set:
        results = retrieve(item["question"], collection, k=k)
        found = any(item["expect"].lower() in r["text"].lower() for r in results)
        hits += found
        if not found:
            misses.append(item["question"])
        top1_sims.append(results[0]["similarity"])
        context_chars.append(sum(len(r["text"]) for r in results))
    n = len(eval_set)
    return {
        "hit_rate"      : hits / n,
        "avg_top1_sim"  : sum(top1_sims) / n,
        "avg_ctx_tokens": int(sum(context_chars) / n / 4),   # ≈ tokens sent to Claude
        "misses"        : misses,
    }

print("SCORECARD — same 5 questions, same scoring, k=3")
print("=" * 66)
print(f"{'chunk size':>10} | {'hit rate':>8} | {'top-1 sim':>9} | {'ctx tokens/question':>19}")
print("-" * 66)
all_scores = {}
for size in CHUNK_SIZES:
    s = score_index(INDEXES[size], EVAL_SET)
    all_scores[size] = s
    print(f"{size:>10} | {s['hit_rate']:>7.0%} | {s['avg_top1_sim']:>9.3f} | {s['avg_ctx_tokens']:>19}")
print("=" * 66)

for size, s in all_scores.items():
    if s["misses"]:
        print(f"\nchunk size {size} MISSED: {s['misses']}")

SCORECARD — same 5 questions, same scoring, k=3
chunk size | hit rate | top-1 sim | ctx tokens/question
------------------------------------------------------------------
       150 |    100% |     0.538 |                 394
       400 |    100% |     0.512 |                 989
       800 |    100% |     0.382 |                1573


📊 **How to read the scorecard** (three columns, three lessons):

- **hit rate** — the number that matters most: how often the *correct* chunk made the
  top-3. On the sample handbook you'll typically see medium (400) at or near 100%,
  with small or large dropping a question — but your PDF may vote differently, and
  **that's a real finding, not a bug**. Look at which question each size missed and
  read the retrieved chunks: too-small chunks separate the question's topic from the
  answer phrase; too-large chunks blur many topics into one lukewarm match.
- **top-1 similarity** — usually *higher* for small chunks even when they miss more.
  Lesson: similarity scores are comparable *within* one index, not across indexes —
  never pick a chunk size because its scores "look higher".
- **ctx tokens/question** — your cost dial. Large chunks send several times more
  tokens per question. At thousands of questions per day, that's real money for —
  as your hit rate shows — not necessarily better answers.

⚠️ **Accuracy note:** 5 questions is a *demo* of evaluation, not an evaluation. Real
teams use 50–200 questions sampled from actual user logs, and re-run the scorecard on
every pipeline change (new chunk size, new model, new prompt). Saying this out loud
in an interview is a seniority signal.

🧪 **Try this:** add a question whose answer sits at the very *end* of a long section —
watch it favour a different chunk size than the others. Then try `k=1` in
`score_index` — watch the hit rates drop and the ranking possibly reshuffle.

### Fourth: hear the difference in Claude's answers

Numbers first, ears second — same question through all three indexes:

In [30]:
TEST_QUESTION = EVAL_SET[1]["question"]     # pick any evaluation question

for size in CHUNK_SIZES:
    print(f"────── chunk size {size} ──────")
    ask_pdf(TEST_QUESTION, collection=INDEXES[size], k=3, show_chunks=True)

────── chunk size 150 ──────
Q: What happens if I ignore a customer dispute?
A: If you ignore a customer dispute on a UPI transaction, the dispute will be automatically decided in the customer's favour [Chunk 4]. Additionally, you will incur a fee of Rs 250 for each lost chargeback [Chunk 4].

To avoid this, you must respond with evidence within 5 working days. Acceptable evidence includes delivery proof, invoices, or customer communication [Chunk 4].
   [tokens in: 500, out: 93]
   [retrieved: #5(0.577), #6(0.562), #4(0.543)]

────── chunk size 400 ──────
Q: What happens if I ignore a customer dispute?
A: If you ignore a customer dispute, the dispute will be decided in the customer's favour automatically. [Chunk 1]

Specifically, when a customer disputes a UPI transaction, you must respond with evidence within 5 working days. If no response is received in time, you lose the dispute by default. [Chunk 1] Additionally, each lost chargeback carries a fee of Rs 250. [Chunk 1]
   [tokens i

📊 **How to read this:** where the correct chunk was retrieved, all three answers
should agree (grounding works). Where a size *missed*, you'll see the honest
"I don't have this information in the document" — usually from the size whose
scorecard row was weakest. One knob, visible consequences, measured. **That was
yesterday's claim; now it's your data.**

🎤 **Interview line you just earned:** *"I tested 150/400/800-token chunks on the same
evaluation set — mid-size chunks won on retrieval hit rate while costing 3–4× less
context than large chunks."* (Quote **your** scorecard, not mine.)

---
# Step 9 — The Grounding Test: Will It Say "I Don't Know"?

🛡️ The last claim to verify: our app *refuses* to answer questions the PDF cannot
answer — instead of hallucinating. Five clearly out-of-scope questions; a grounded
app should decline **all five**. (Because Rule 2 pins the exact refusal sentence,
we can detect refusals in code — that design choice is what makes grounding
*measurable*.)

In [31]:
out_of_scope = [
    "What is the capital of France?",
    "Who won the 2011 Cricket World Cup?",
    "What is the boiling point of water?",
    "Write a poem about the monsoon.",
    "What is Apple's current stock price?",
]

refused = 0
for q in out_of_scope:
    answer = ask_pdf(q, show_chunks=False)
    if "don't have this information" in answer.lower():
        refused += 1

print("=" * 50)
print(f"GROUNDING SCORE: {refused}/{len(out_of_scope)} correctly refused")
print("✅ Solid grounding" if refused >= 4 else "⚠️ Leaky grounding — tighten the rules in GROUNDING_PROMPT")

Q: What is the capital of France?
A: I don't have this information in the document.

The document provided only contains NammaPay Merchant Policy information about settlements, refunds, support, and fees. It does not include information about geography or capital cities.
   [tokens in: 1033, out: 49]

Q: Who won the 2011 Cricket World Cup?
A: I don't have this information in the document.

The provided context only contains information about NammaPay's merchant policies, including details about refunds, disputes, chargebacks, KYC requirements, and fees. It does not contain any information about the Cricket World Cup.
   [tokens in: 955, out: 61]

Q: What is the boiling point of water?
A: I don't have this information in the document.

The document provided is a NammaPay Merchant Policy Handbook that covers payment fees, chargeback policies, KYC requirements, and merchant support details. It does not contain information about the boiling point of water.
   [tokens in: 1015, out: 61]

Q:

📊 **How to read this:** expect **5/5** (4/5 is acceptable). Note what happened
under the hood: retrieval *still returned* the top-3 least-bad chunks for "capital of
France" — retrieval never says no. The refusal came entirely from the **grounding
prompt**. Retrieval finds; grounding disciplines.

🧪 **Try this (break it on purpose):** delete Rule 2 (the explicit refusal
instruction) from `GROUNDING_PROMPT`, re-run the cell, and watch the score drop as
Claude starts helpfully answering geography questions "since it knows them". Put the
rule back. You will never forget why it's there. Also try a *near-miss* question —
about your PDF's topic but not answered in it — those are far harder to refuse than
"capital of France", and where real systems leak.

---
# 🏗️ What You Built — and What's Missing Before Production

```mermaid
flowchart LR
    subgraph OFFLINE["OFFLINE - ran once"]
        A[your PDF] --> B[pdfplumber extract] --> C[chunk +overlap] --> D[MiniLM embed] --> E[(ChromaDB cosine)]
    end
    subgraph ONLINE["ONLINE - every ask_pdf call"]
        Q[question] --> F[embed question] --> G[top-3 retrieve] --> H[Claude Haiku + grounding] --> I[cited answer or honest refusal]
    end
    E --> G
```

**Production gap list** (name these in any design review):
persistent storage (`chromadb.PersistentClient` instead of in-memory — today's index
dies with the runtime); re-indexing when documents change; OCR for scanned PDFs;
access control + audit logs; a 50–200 question evaluation set running on every
change; cost/latency monitoring; escalation to humans on refusals; and hybrid
retrieval (BM25 + embeddings) when users search by exact codes and names.

# 📝 Session Summary

You built a complete PDF Q&A app on the live Claude API: pdfplumber extraction (with
per-page character counts to catch scanned pages), sentence-aware chunking with
overlap, MiniLM embeddings, ChromaDB in explicit cosine space, top-3 retrieval, and
grounded generation with Claude Haiku that cites chunk numbers and refuses
out-of-scope questions with a fixed, detectable sentence. Then you did what separates
engineers from tutorial-followers: you *measured* — three chunk sizes indexed from
the same PDF, scored with one function on one evaluation set, revealing the real
trade-off between retrieval hit rate and context cost; and a grounding test proving
the app says "I don't know" instead of hallucinating. The numbers on your screen are
yours to quote.

# ✅ What You Learned Today

You can now:
- [ ] Extract and clean text from any text PDF (and detect scanned ones)
- [ ] Chunk with overlap, embed with Sentence Transformers, store in ChromaDB
- [ ] Build a grounded, cited Q&A function on Claude Haiku
- [ ] Design a small evaluation set (user-language questions, document-language expects)
- [ ] Compare chunk sizes fairly — one eval set, one scoring function, one scorecard
- [ ] Prove grounding works with an out-of-scope refusal test
- [ ] Read token counts as money, and argue chunk-size choices with data

# 🗂️ Cheat Sheet — the whole app in 8 lines

```python
chunks  = chunk_document(text, chunk_tokens=400)                 # ① chunk
vecs    = EMBEDDER.encode(chunks)                                # ② embed
col     = build_collection("docs", chunks, vecs)                 # ③ store (cosine!)
hits    = retrieve(question, col, k=3)                           # ④ retrieve
context = "\n\n".join(f"[Chunk {h['chunk_index']}]\n{h['text']}" for h in hits)
reply   = client.messages.create(model=MODEL, max_tokens=400,    # ⑤ generate
              system=GROUNDING_PROMPT.format(context=context),
              messages=[{"role": "user", "content": question}])
```

**Numbers from today worth remembering:** MiniLM = 384 dims, ~90 MB · chunk sizes
tested 150/400/800 tokens · k=3 · refusal target ≥4/5 · eval sets in production:
50–200 questions.

# ⏱️ 5-Minute Revision Guide

1. Extract text per page; print character counts — zero chars = scanned page.
2. Clean text: fix hyphen line-breaks, collapse spaces and blank lines.
3. Chunk by whole sentences; carry the last sentence forward as overlap.
4. `chunk_tokens` is a parameter, because you will tune it — with data.
5. Embed chunks AND questions with the same model (MiniLM here).
6. ChromaDB: set `{"hnsw:space": "cosine"}` explicitly; similarity = 1 − distance.
7. Delete-and-recreate collections in notebooks; upsert by stable IDs in real apps.
8. Grounding prompt: only-from-context + exact refusal sentence + chunk citations.
9. A fixed refusal sentence makes grounding *measurable* in code.
10. Evaluation set: user-language questions + document-language expected phrases.
11. One scoring function for all contenders — else the comparison is meaningless.
12. Hit rate beats similarity: scores aren't comparable across different indexes.
13. Large chunks cost several × more context tokens per question.
14. Retrieval never refuses; refusal comes from the grounding prompt.
15. 5 questions demo the method; production needs 50–200 from real logs.
16. In-memory Chroma dies with the runtime; persistence is a one-line change.

# 🎤 Interview Preparation Notes

**Q1. How would you evaluate a RAG system?**
"Two layers. Retrieval: an eval set of real user questions with known correct
chunks — measure hit rate / recall@k. Generation: grounded-answer quality and,
critically, correct refusals on out-of-scope questions. Small fixed sets to start,
grown from production logs, re-run on every change like unit tests."

**Q2. What did changing chunk size actually change in your experiment?**
"Three things: number of chunks (inverse), retrieval hit rate (mid-size won on my
set — small chunks split answers from their context, large ones blurred topics), and
context tokens per question (large ≈ several × cost of small). I'd tune it per
document type, by scorecard."

**Q3. Your PDF bot hallucinated in a client demo. Fix order?**
"(1) Check retrieval for that question — was the right chunk even in the top-k? If
not, it's a retrieval problem: fix chunking/k/model. (2) If context was right,
tighten grounding — explicit refusal exit, required citations. (3) Add the failing
question to the eval set so the regression is caught forever. Prompt-tweaking first
is the junior mistake — diagnose retrieval before generation."

**Q4. Why Haiku and not the biggest model?**
"Grounded Q&A is a reading task, not a reasoning marathon — the knowledge arrives in
the context. Haiku is fast and cheap, and the pipeline's quality bottleneck is
retrieval, not model size. I'd escalate model size only if evaluation shows
generation — not retrieval — is what's failing."

# 📚 Assignment

**Beginner** — Run the whole lab on a different PDF. Rewrite the 5 evaluation
questions and re-print the scorecard. Did the same chunk size win?

**Intermediate** — Add a 4th contender (say 250 tokens) and an *overlap experiment*:
chunk 400 with `overlap_sentences=0` vs `2`. Which questions change outcome?

**Advanced** — Make it persistent and incremental: switch to
`chromadb.PersistentClient(path="./db")`, support adding a *second* PDF into the same
collection with a `source` metadata field, and add a `where={"source": ...}` filter
to `retrieve` so users can ask one document or all.

**Project** — Ship "Meera's assistant": wrap `ask_pdf` in a simple Gradio chat UI,
show citations under each answer, log every (question, chunks, answer) to a JSONL
file, and write a 5-line README explaining your chunk-size choice — with your
scorecard as evidence.

# 🧪 Assessment

**Part A — Multiple choice (10)**

1. The per-page character counts in Step 3 exist mainly to catch:
   a) slow PDFs b) scanned pages that extract no text c) copyright issues d) large files
2. Our chunker carries the last sentence into the next chunk. This implements:
   a) semantic chunking b) overlap c) re-ranking d) compression
3. `chunk_tokens * 4` converts:
   a) tokens→words b) tokens→approximate characters c) characters→tokens d) tokens→bytes
4. We embed the question with:
   a) Claude Haiku b) the same MiniLM model as the chunks c) any model d) BM25
5. `{"hnsw:space": "cosine"}` is required because:
   a) Chroma has no default b) Chroma defaults to L2 distance c) cosine is faster d) Claude requires it
6. With cosine space, similarity is computed from Chroma's distance as:
   a) 1/(1+distance) b) 1 − distance c) distance² d) −distance
7. In the scorecard, the most decision-relevant column is:
   a) top-1 similarity b) hit rate c) ctx tokens d) chunk count
8. Small chunks showed higher top-1 similarity but sometimes lower hit rate because:
   a) similarity is comparable across indexes b) scores within different indexes aren't comparable; focused chunks can split answer from topic c) MiniLM fails on small text d) Chroma bugs
9. The grounding refusal is detectable in code because:
   a) Claude sets a refusal flag b) the prompt pins an exact refusal sentence c) tokens go to zero d) Chroma reports it
10. For "What is the capital of France?", retrieval:
    a) returns nothing b) errors c) still returns the least-bad top-3 chunks d) asks Claude first

**Part B — Short answer (5)**

11. Why must all three chunk sizes be scored by the same function on the same questions?
12. Why write eval questions in "user language" but expects in "document language"?
13. Your app refuses a question whose answer IS in the PDF. Name the two most likely
    stages at fault and how you'd tell them apart.
14. What breaks when the Colab runtime restarts, and what's the one-line fix direction?
15. Why does `build_collection` delete the collection before creating it?

**Part C — Scenario (3)**

16. The scorecard shows: 150→60% hit rate, 400→100%, 800→80%; ctx tokens 310/1150/2350.
    The client wants "the most accurate AND cheapest" setup. Your recommendation and
    the one-sentence justification you'd put on a slide?
17. A teammate reports "grounding is broken — it answered a general-knowledge
    question". You re-run the grounding test: 5/5 refused. What do you ask for next,
    and what are the two most likely explanations?
18. After adding 50 more PDFs to one collection, answers start citing the wrong
    documents. Which two features you built today (but maybe didn't use) solve this,
    and how?

# 🔑 Answer Key

**MCQ:** 1-b · 2-b · 3-b · 4-b · 5-b · 6-b · 7-b · 8-b · 9-b · 10-c

**Short answers (gist):**
11. Changing either the questions or the scoring per contender destroys comparability —
identical measurement is the definition of a fair experiment.
12. That mirrors reality: users phrase questions colloquially; documents phrase answers
formally. The gap between the two is exactly what embeddings must bridge, so the eval
must contain that gap.
13. Retrieval (right chunk missed top-k) vs generation (chunk present, Claude still
refused). Tell apart by printing the retrieved chunks: if the answer text isn't there,
it's retrieval; if it is, tighten/inspect the grounding prompt.
14. The in-memory ChromaDB index vanishes (and uploaded files too). Fix direction:
`chromadb.PersistentClient(path=...)` + re-usable stored PDFs.
15. Re-running the cell would otherwise fail ("already exists") or duplicate records;
delete-and-recreate keeps the notebook idempotent. Production uses upserts instead.

**Scenarios:**
16. Recommend 400: it wins accuracy outright (100%) at roughly half the token cost of
800; 150 is cheapest but wrong 40% of the time — "the cheapest wrong answer is still
wrong". Slide line: "400-token chunks: highest measured hit rate at ~½ the context
cost of the runner-up."
17. Ask for the exact failing question and the logged retrieved chunks. Likely: (a) it
was a *near-miss* question — on-topic but unanswered in the PDF, which is harder to
refuse than clearly foreign questions; or (b) prompt drift — the deployed
GROUNDING_PROMPT differs from the tested one. (Both are why you log question+chunks+answer.)
18. Metadata + filtering: you stored `source` per chunk — add `where={"source": ...}`
filters to `retrieve`; and cite source file alongside chunk number in the context
labels so Claude's citations carry the document name.

# 🔗 Sources

- Anthropic docs — Messages API, model IDs, token usage: docs.claude.com
- ChromaDB docs — collections, distance metrics, PersistentClient: docs.trychroma.com
- Sentence-Transformers — all-MiniLM-L6-v2 model card: sbert.net
- pdfplumber documentation: github.com/jsvine/pdfplumber
- Lewis et al. (2020), the original RAG paper: arxiv.org/abs/2005.11401

---
🎓 **You now hold the two halves:** yesterday you could *explain* RAG to anyone;
today you can *prove* it with a working app and a scorecard. That combination —
plain-language explanation + measured evidence — is exactly what presenting this
topic confidently looks like.